In [23]:
import pandas as pd
import numpy as np
import ftfy
import random
import torch
import json
import plotly.express as px 

from collections import Counter
from collections import defaultdict
from datasets import Dataset, DatasetDict
from preprocess import transcriptions
from transformers import AutoTokenizer
from transformers import AutoModelForTokenClassification
from torch.utils.data import DataLoader
from sklearn.metrics import classification_report


## Préparation des données

In [2]:
# Chargement des données et gestion des problèmes d'encodage
annotations1 = pd.read_csv("annotations/annotation_token_strategy_annotations.csv", sep=";",encoding="latin-1")
annotations1 = annotations1.apply(lambda col: col.map(lambda x: ftfy.fix_text(x) if isinstance(x, str) else x))
annotations1 = annotations1.dropna()

annotations2 = pd.read_csv("annotations/annotation_tokens_complémentaire.csv", sep=";")
annotations2 = annotations2.rename(columns = {"token":"token_text", 
                                    'Unnamed: 0':"phrase_id", 
                                    "text_id":"doc_id_origine"})
annotations2 = annotations2.dropna()
annotations2["label_concret"] = annotations2["label_concret"].replace('B-VERB_CINCRET', 'B-VERB_CONCRET')

# Erreurs de frappe constatées a posteriori
annotations1["label_concret"] = annotations1["label_concret"].replace('B-VERB_CONRET', 'B-VERB_CONCRET')
annotations1["label_concret"] = annotations1["label_concret"].replace('B-VERB_CONCRET ', 'B-VERB_CONCRET')
annotations1["label_concret"] = annotations1["label_concret"].replace('I-VERB-CONCRET', 'I-VERB_CONCRET')
annotations1["label_concret"] = annotations1["label_concret"].replace('E-VERB-CONCRET', 'E-VERB_CONCRET')
annotations1["label_concret"] = annotations1["label_concret"].replace('B-VERB-CONTEXTE', 'B-VERB_CONTEXTE') 

annotations1["label_discours"] = annotations1["label_discours"].replace('B-ACT', 'B-VERB_ACT')  
annotations1["label_discours"] = annotations1["label_discours"].replace('I-VERB_ETA', 'I-VERB_ETAT') 
annotations1["label_discours"] = annotations1["label_discours"].replace('V-VERB_ACT', 'B-VERB_ACT')

textes_annotes = pd.unique(annotations1["doc_id_origine"])  # A modifier


In [3]:
# Création du df de travail pour le label_concret
annotations_work_concret1 = annotations1[["phrase_id", "token_text", "label_concret"]]
annotations_work_concret2 = annotations2[["phrase_id", "token_text", "label_concret"]]

# Clé identifiant le fichier d'origine
annotations_work_concret1['_src'] = 0
annotations_work_concret2['_src'] = 1

df_concat_concret = pd.concat([annotations_work_concret1, annotations_work_concret2], ignore_index=True)

# Renuméroter les 'phrase_id' en tenant compte de la source
df_concat_concret['phrase_id'] =(
    df_concat_concret.groupby(['_src', 'phrase_id'], sort=False)
    .ngroup() + 1
)

annotations_work_concret = df_concat_concret.groupby("phrase_id").agg(
    token_text=("token_text", list),
    label_concret=("label_concret", lambda x: [v[0] if isinstance(v, list) else v for v in x])
).reset_index()

In [4]:
annotations_work_concret = annotations_work_concret.drop_duplicates(subset="token_text")
print(annotations_work_concret.shape)

(628, 3)


In [5]:
# Création du df de travail pour le label_discours
annotations_work_disc1 = annotations1[["phrase_id", "token_text", "label_discours"]]
annotations_work_disc2 = annotations2[["phrase_id", "token_text", "label_discours"]]

# Clé identifiant le fichier d'origine
annotations_work_disc1['_src'] = 0
annotations_work_disc2['_src'] = 1

df_concat_disc = pd.concat([annotations_work_disc1, annotations_work_disc2], ignore_index=True)

# Renuméroter les 'phrase_id' en tenant compte de la source
df_concat_disc['phrase_id'] = (
    df_concat_disc.groupby(['_src', 'phrase_id'], sort=False)
    .ngroup() + 1
)
annotations_work_disc = df_concat_disc.groupby("phrase_id").agg(
    token_text=("token_text", list),
    label_discours=("label_discours", lambda x: [v[0] if isinstance(v, list) else v for v in x])
).reset_index()

In [6]:
annotations_work_disc = annotations_work_disc.drop_duplicates(subset="token_text")
print(annotations_work_disc.shape)

(628, 3)


In [7]:
# Préparation pour avoir une partie train et une partie test

def split_by_phrase_id(df, col_tag, train_ratio=0.8, val_ratio=0.1, seed=42):
    
    sentences = df.to_dict(orient="records")

    random.seed(seed)
    random.shuffle(sentences)

    n = len(sentences)

    def density_bucket(example):
        n_entities = sum(1 for t in example[col_tag] if t.startswith("B-"))
        if n_entities == 0:
            return 0
        elif n_entities <= 2:
            return 1
        else:
            return 2

    buckets = {0: [], 1: [], 2: []}
    for _, ex in df.iterrows():
        buckets[density_bucket(ex)].append(ex)

    train, val, test = [], [], []

    for bucket_data in buckets.values():
        random.shuffle(bucket_data)
        n = len(bucket_data)
        n_train = int(n * train_ratio)
        n_val   = int(n * val_ratio)

        train += bucket_data[:n_train]
        val   += bucket_data[n_train:n_train + n_val]
        test  += bucket_data[n_train + n_val:]

    random.shuffle(train)
    random.shuffle(val)
    random.shuffle(test)

    print(f"Train : {len(train)} phrases")
    print(f"Val : {len(val)} phrases")
    print(f"Test : {len(test)} phrases")

    return train, val, test

## Travail sur le label concret

In [9]:
train, val, test = split_by_phrase_id(annotations_work_concret, "label_concret")

Train : 476 phrases
Val   : 58 phrases
Test  : 63 phrases


In [10]:
labels_conc = list({t for row in annotations_work_concret["label_concret"] for t in row})
num_labels_conc = len(labels_conc)
id2label = {id:label for id, label in enumerate(labels_conc)}
label2id = {label:id  for id, label in enumerate(labels_conc)}

def encode_labels(example, col_tag):
    example[col_tag] = [label2id[t] for t in example[col_tag]]
    return example

COL_TAG = "label_concret"
train = [row.to_dict() for row in train]
test  = [row.to_dict() for row in test]
val   = [row.to_dict() for row in val]

train_processed = [encode_labels(ex, COL_TAG) for ex in train]
test_processed = [encode_labels(ex, COL_TAG) for ex in test]
val_processed = [encode_labels(ex, COL_TAG) for ex in val]

train_dataset = Dataset.from_list(train_processed)
test_dataset = Dataset.from_list(test_processed)
val_dataset = Dataset.from_list(val_processed)

dataset = DatasetDict({"train": train_dataset, "test":test_dataset, "validation": val_dataset})

In [ ]:
from transformers import AutoTokenizer
from transformers import AutoModelForTokenClassification

MODEL_NAME = "camembert-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_and_align_labels(examples):
    tokenized = tokenizer(
        examples["token_text"],
        col_tag = COL_TAG,
        truncation=True,
        is_split_into_words=True,
        padding="max_length",
        max_length=128,
    )

    all_labels = []
    for i, labels in enumerate(examples[COL_TAG]):
        word_ids = tokenized.word_ids(batch_index=i)
        print(f"Nb labels: {len(labels)}, max word_id: {max(w for w in word_ids if w is not None)}")
        aligned_labels = []
        prev_word_id = None

        for word_id in word_ids:
            if word_id is None:
                aligned_labels.append(-100)
            elif word_id != prev_word_id:
                aligned_labels.append(labels[word_id])
            else:
                aligned_labels.append(-100)
            prev_word_id = word_id

        all_labels.append(aligned_labels)

    tokenized["labels"] = all_labels
    return tokenized

tokenized_dataset = dataset.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=dataset["train"].column_names,
)

In [30]:
from transformers import (
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
)
import evaluate

model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(labels_conc),
    id2label=id2label,
    label2id=label2id
)

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 3613.52it/s]
CamembertForTokenClassification LOAD REPORT from: camembert-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.bias          | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
classifier.weight           | MISSING    | 
classifier.bias             | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [17]:
def compute_metrics(eval_preds):
    logits, labels = eval_preds
    predictions = np.argmax(logits, axis=-1)

    true_preds, true_labels = [], []
    for pred_seq, label_seq in zip(predictions, labels):
        p_row, l_row = [], []
        for p, l in zip(pred_seq, label_seq):
            if l != -100: 
                p_row.append(id2label[p])
                l_row.append(id2label[l])
        true_preds.append(p_row)
        true_labels.append(l_row)

    results = seqeval.compute(predictions=true_preds, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall":    results["overall_recall"],
        "f1":        results["overall_f1"],
        "accuracy":  results["overall_accuracy"],
    }

In [ ]:
import torch
from collections import Counter
# Métrique seqeval (F1 par entité)
seqeval = evaluate.load("seqeval")

all_labels_flat = [t for ex in train for t in ex["label_concret"]]
counts = Counter(all_labels_flat)
total  = sum(counts.values())
weights = torch.zeros(num_labels_conc)
for label_id, count in counts.items():
    weights[label_id] = total / (num_labels_conc * count)
print("Poids par classe :", {id2label[i]: round(w.item(), 2) for i, w in enumerate(weights)})
weights = torch.clamp(weights, max=10.0)

class WeightedTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights 

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = torch.nn.CrossEntropyLoss(
            weight=self.class_weights.to(logits.device),
            ignore_index=-100
        )
        loss = loss_fct(logits.view(-1, num_labels_conc), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

# Collator : gère le padding dynamique
data_collator = DataCollatorForTokenClassification(tokenizer)

training_args = TrainingArguments(
    output_dir="./camembert-CONC",
    num_train_epochs=5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=1e-4,
    warmup_ratio = 0.1,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_dir="./logs",
    logging_steps=1,
    fp16=torch.cuda.is_available(),
)

trainer = WeightedTrainer(
    model=model,
    class_weights = weights,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

trainer.train()

# Sauvegarde du meilleur modèle
trainer.save_model("./camembert-final-CONC")
tokenizer.save_pretrained("./camembert-final-CONC")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Poids par classe : {'E-VERB_CONCRET': 28.99, 'B-VERB_CONCRET': 6.68, 'I-VERB_CONCRET': 11.85, 'B-VERB_CONTEXTE': 5.26, 'B-OBJ': 16.82, 'B-SUJ': 3.14, 'I-VERB_CONTEXTE': 20.48, 'O': 0.12, 'E-VERB_CONTEXTE': 47.11}


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,1.124766,1.190013,0.302326,0.764706,0.433333,0.812606
2,0.828480,0.846374,0.387574,0.770588,0.515748,0.837024
3,0.707753,0.741681,0.427046,0.705882,0.532151,0.882453
4,0.388936,0.671216,0.639269,0.823529,0.719794,0.944350
5,0.445630,0.653343,0.673077,0.823529,0.740741,0.949461


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.09s/it]
There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.la

('./camembert-final-CONC/tokenizer_config.json',
 './camembert-final-CONC/tokenizer.json')

In [ ]:
# Courbes d'apprentissage du modèle
with open("./camembert-CONC/checkpoint-300/trainer_state.json", "r") as file: 
    training_state = json.load(file)

loss = []

for log in training_state ["log_history"]:
    step = log["step"]
    if "loss" in log:
        loss += [{"step": step, "loss": log["loss"], "split": "train"}]
    elif "eval_loss" in log:
        loss += [{"step": step, "loss": log["eval_loss"], "split": "eval"}]
    else: 
        # thweird
        print(log)

loss = pd.DataFrame(loss)

px.line(loss, x = "step", y = "loss", color =  "split") 

In [ ]:
labels_true = []
labels_pred = []

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()

loader = DataLoader(
    tokenized_dataset["test"].with_format("torch"),
    batch_size=8,
    collate_fn=data_collator,
    shuffle=False,
)

for batch in loader:

    # Puis dans ta boucle d'inférence
    model_input = {
    "input_ids": batch["input_ids"].detach().clone().to(device),
    "attention_mask": batch["attention_mask"].detach().clone().to(device),}

    with torch.no_grad():
        logits = model(**model_input).logits

    predictions = torch.argmax(logits, dim=-1).cpu().numpy()
    gold_labels = np.array(batch["labels"])

    for pred_seq, label_seq in zip(predictions, gold_labels):
        for p, l in zip(pred_seq, label_seq):
            if l != -100: 
                labels_pred.append(id2label[int(p)])
                labels_true.append(id2label[int(l)])

(
    pd.DataFrame({
        "predict": labels_pred,
        "gold_standard": labels_true,
    })
    .to_csv("./outputs/predictions_conc.csv", index=False)
)

/tmp/ipykernel_16232/1416306588.py:21: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  "input_ids": torch.tensor(batch["input_ids"]).to(device),
/tmp/ipykernel_16232/1416306588.py:22: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  "attention_mask": torch.tensor(batch["attention_mask"]).to(device),}
/tmp/ipykernel_16232/1416306588.py:28: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  gold_labels = np.array(batch["labels"])


In [ ]:
print(classification_report(y_true = labels_true, y_pred = labels_pred))

                 precision    recall  f1-score   support

          B-OBJ       0.31      0.89      0.46         9
          B-SUJ       0.91      1.00      0.95        79
 B-VERB_CONCRET       0.77      0.77      0.77        35
B-VERB_CONTEXTE       0.72      0.90      0.80        49
 E-VERB_CONCRET       0.33      0.88      0.48         8
E-VERB_CONTEXTE       0.31      0.50      0.38         8
 I-VERB_CONCRET       0.44      0.70      0.54        23
I-VERB_CONTEXTE       0.45      0.94      0.61        18
              O       1.00      0.95      0.97      1917

       accuracy                           0.94      2146
      macro avg       0.58      0.84      0.66      2146
   weighted avg       0.96      0.94      0.95      2146



Meilleurs paramètres pour le moment :
training_args = TrainingArguments(
    output_dir="./camembert",
    num_train_epochs=5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=1e-4,
    warmup_ratio = 0.1,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_dir="./logs",
    logging_steps=1,
    fp16=False,
)

In [37]:
model = AutoModelForTokenClassification.from_pretrained("./camembert-final-CONC")
tokenizer = AutoTokenizer.from_pretrained("./camembert-final-CONC")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3243.82it/s]


CamembertForTokenClassification(
  (dropout): Dropout(p=0.1, inplace=False)
  (classifier): Linear(in_features=768, out_features=9, bias=True)
  (roberta): CamembertModel(
    (embeddings): CamembertEmbeddings(
      (word_embeddings): Embedding(32005, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
    )
    (encoder): CamembertEncoder(
      (layer): ModuleList(
        (0-11): 12 x CamembertLayer(
          (attention): CamembertAttention(
            (self): CamembertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
         

In [38]:
def annotate_text(text, model, tokenizer, max_length=512, stride=128):

    encoding = tokenizer(
        text,
        return_offsets_mapping=True,
        return_overflowing_tokens=True,
        truncation=True,
        max_length=max_length,
        stride=stride,
        padding="max_length",
        return_tensors="pt",
    )

    offset_mapping = encoding.pop("offset_mapping")
    encoding.pop("overflow_to_sample_mapping")

    num_labels  = model.config.num_labels
    char_scores = np.zeros((len(text), num_labels), dtype=np.float32)
    char_counts = np.zeros(len(text), dtype=np.int32)

    for i in range(len(encoding["input_ids"])):
        chunk = {k: v[i].unsqueeze(0).to(device) for k, v in encoding.items()}
        with torch.no_grad():
            logits = model(**chunk).logits[0].cpu()
        probs = torch.softmax(logits, dim=-1).numpy()

        for token_idx, (start, end) in enumerate(offset_mapping[i]):
            s, e = int(start), int(end)
            if s == e:
                continue
            char_scores[s:e] += probs[token_idx]
            char_counts[s:e] += 1

    mask = char_counts > 0
    char_scores[mask] /= char_counts[mask, np.newaxis]

    import re
    id2label = model.config.id2label
    results  = []

    for m in re.finditer(r'\S+', text):
        s, e   = m.start(), m.end()
        window = char_scores[s:e]
        counts = char_counts[s:e]

        if counts.sum() == 0:
            label_id, score = 0, 0.0
        else:
            avg = window.mean(axis=0)
            label_id = int(avg.argmax())
            score = float(avg[label_id])

        results.append({
            "word": text[s:e],
            "label": id2label[label_id],
            "score": round(score, 4),
            "start": s,
            "end": e,
        })

    return results
 

In [42]:
import json
from tqdm import tqdm

def annotate_corpus(texts, ids, output_path):

    with open(output_path, "w", encoding="utf-8") as fout:
        for doc_id, text in tqdm(zip(ids, texts), total=len(texts)):
            try:
                annotations = annotate_text(text, model, tokenizer)
                record = {
                    "id": doc_id,
                    "text": text,
                    "annotations": annotations
                }
            except Exception as e:
                record = {"id": doc_id, "error": str(e)}

            fout.write(json.dumps(record, ensure_ascii=False) + "\n")

    print(f"Sauvegardé dans {output_path}")



In [43]:
textes = transcriptions["texte_nettoye"].to_list()
ids = transcriptions["id"]
output_path = "outputs/predictions_tot.jsonl"

annotate_corpus(textes, ids, output_path)

100%|██████████| 5936/5936 [12:50<00:00,  7.70it/s]

Sauvegardé dans outputs/predictions_tot.jsonl


## Travail sur le label_discours

In [12]:
train, val, test = split_by_phrase_id(annotations_work_disc, "label_discours")

Train : 501 phrases
Val : 61 phrases
Test : 66 phrases


In [13]:
labels_disc = list({t for row in annotations_work_disc["label_discours"] for t in row})
num_labels_disc = len(labels_disc)
id2label = {id:label for id, label in enumerate(labels_disc)}
label2id = {label:id  for id, label in enumerate(labels_disc)}

def encode_labels(example, col_tag):
    example[col_tag] = [label2id[t] for t in example[col_tag]]
    return example

COL_TAG = "label_discours"
train = [row.to_dict() for row in train]
test = [row.to_dict() for row in test]
val = [row.to_dict() for row in val]

train_processed = [encode_labels(ex, COL_TAG) for ex in train]
test_processed = [encode_labels(ex, COL_TAG) for ex in test]
val_processed = [encode_labels(ex, COL_TAG) for ex in val]

train_dataset = Dataset.from_list(train_processed)
test_dataset = Dataset.from_list(test_processed)
val_dataset = Dataset.from_list(val_processed)

dataset = DatasetDict({"train": train_dataset, "test":test_dataset, "validation": val_dataset})

In [14]:
MODEL_NAME = "camembert-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_and_align_labels(examples):
    tokenized = tokenizer(
        examples["token_text"],
        col_tag = COL_TAG,
        truncation=True,
        is_split_into_words=True,
        padding="max_length",
        max_length=128,
    )

    all_labels = []
    for i, labels in enumerate(examples[COL_TAG]):
        word_ids = tokenized.word_ids(batch_index=i)
        # print(f"Nb labels: {len(labels)}, max word_id: {max(w for w in word_ids if w is not None)}")
        aligned_labels = []
        prev_word_id = None

        for word_id in word_ids:
            if word_id is None:
                aligned_labels.append(-100)
            elif word_id != prev_word_id:
                aligned_labels.append(labels[word_id])
            else:
                aligned_labels.append(-100)
            prev_word_id = word_id

        all_labels.append(aligned_labels)

    tokenized["labels"] = all_labels
    return tokenized

tokenized_dataset = dataset.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=dataset["train"].column_names,
)

Map: 100%|██████████| 61/61 [00:00<00:00, 4088.47 examples/s]


In [38]:
from transformers import (
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
)
import evaluate

model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(labels_disc),
    id2label=id2label,
    label2id=label2id
)

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 4054.34it/s]
[transformers] CamembertForTokenClassification LOAD REPORT from: camembert-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
classifier.weight           | MISSING    | 
classifier.bias             | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [39]:
seqeval = evaluate.load("seqeval")

all_labels_flat = [t for ex in train for t in ex["label_discours"]]
counts = Counter(all_labels_flat)
total  = sum(counts.values())
weights = torch.zeros(num_labels_disc)
for label_id, count in counts.items():
    weights[label_id] = total / (num_labels_disc * count)
print("Poids par classe :", {id2label[i]: round(w.item(), 2) for i, w in enumerate(weights)})
weights = torch.clamp(weights, max=10.0)

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = torch.nn.CrossEntropyLoss(
            weight=weights.to(logits.device),
            ignore_index=-100
        )
        loss = loss_fct(logits.view(-1, num_labels_disc), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

# Collator : gère le padding dynamique
data_collator = DataCollatorForTokenClassification(tokenizer)

training_args = TrainingArguments(
    output_dir="./camembert-DISC",
    num_train_epochs=5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=1e-4,
    warmup_ratio = 0.1,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_dir="./logs",
    logging_steps=1,
    fp16=False,
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

trainer.train()

# Sauvegarde du meilleur modèle
trainer.save_model("./camembert-final-DISC")
tokenizer.save_pretrained("./camembert-final-DISC")

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Poids par classe : {'B-OBJ': 14.47, 'E-VERB_ETAT': 81.33, 'B-SUJ': 2.71, 'I-VERB_ETAT': 25.12, 'B-VERB_ETAT': 6.91, 'I-VERB_ACT': 17.25, 'B-VERB_ACT': 4.08, 'B_VERB-ETAT': 1707.9, 'O': 0.11, 'E-VERB_ACT': 46.16}


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,1.298804,1.036174,0.514706,0.838323,0.637813,0.927083
2,0.624326,0.742145,0.441606,0.724551,0.548753,0.901786
3,0.544539,0.659594,0.525641,0.736527,0.613466,0.929563
4,0.415678,0.642608,0.658163,0.772455,0.710744,0.955357
5,0.314100,0.653006,0.691489,0.778443,0.732394,0.962302


Writing model shards: 100%|██████████| 1/1 [00:02<00:00,  2.05s/it]


('./camembert-final-DISC/tokenizer_config.json',
 './camembert-final-DISC/tokenizer.json')

In [40]:
with open("./camembert-DISC/checkpoint-315/trainer_state.json", "r") as file: 
    training_state = json.load(file)

loss = []

for log in training_state ["log_history"]:
    step = log["step"]
    if "loss" in log:
        loss += [{"step": step, "loss": log["loss"], "split": "train"}]
    elif "eval_loss" in log:
        loss += [{"step": step, "loss": log["eval_loss"], "split": "eval"}]
    else: 
        # thweird
        print(log)

loss = pd.DataFrame(loss)

px.line(loss, x = "step", y = "loss", color =  "split") 

In [41]:
labels_true = []
labels_pred = []

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()

for batch in tokenized_dataset["test"].batch(batch_size=4, drop_last_batch=False):

    # Puis dans ta boucle d'inférence
    model_input = {
    "input_ids": torch.tensor(batch["input_ids"]).to(device),
    "attention_mask": torch.tensor(batch["attention_mask"]).to(device),}

    with torch.no_grad():
        logits = model(**model_input).logits

    predictions = torch.argmax(logits, dim=-1).cpu().numpy()
    gold_labels = np.array(batch["labels"])

    for pred_seq, label_seq in zip(predictions, gold_labels):
        for p, l in zip(pred_seq, label_seq):
            if l != -100:
                labels_pred.append(id2label[int(p)])
                labels_true.append(id2label[int(l)])

(
    pd.DataFrame({
        "predict": labels_pred,
        "gold_standard": labels_true,
    })
    .to_csv("./outputs/predictions_disc.csv", index=False)
)

print(classification_report(y_true = labels_true, y_pred = labels_pred))

Batching examples: 100%|██████████| 66/66 [00:00<00:00, 1105.91 examples/s]


              precision    recall  f1-score   support

       B-OBJ       0.50      1.00      0.67        12
       B-SUJ       0.99      0.99      0.99        94
  B-VERB_ACT       0.87      0.89      0.88        66
 B-VERB_ETAT       0.64      0.82      0.72        34
  E-VERB_ACT       0.10      0.20      0.13         5
 E-VERB_ETAT       0.00      0.00      0.00         4
  I-VERB_ACT       0.24      0.46      0.32        13
 I-VERB_ETAT       0.35      0.47      0.40        15
           O       0.99      0.98      0.98      2257

    accuracy                           0.96      2500
   macro avg       0.52      0.65      0.57      2500
weighted avg       0.97      0.96      0.97      2500



Meilleurs paramètres :
training_args = TrainingArguments(
    output_dir="./camembert-DISC",
    num_train_epochs=5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=1e-4,
    warmup_ratio = 0.1,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_dir="./logs",
    logging_steps=1,
    fp16=False,
)

In [42]:
model = AutoModelForTokenClassification.from_pretrained("./camembert-final-DISC")
tokenizer = AutoTokenizer.from_pretrained("./camembert-final-DISC")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2608.32it/s]


CamembertForTokenClassification(
  (dropout): Dropout(p=0.1, inplace=False)
  (classifier): Linear(in_features=768, out_features=10, bias=True)
  (roberta): CamembertModel(
    (embeddings): CamembertEmbeddings(
      (word_embeddings): Embedding(32005, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
    )
    (encoder): CamembertEncoder(
      (layer): ModuleList(
        (0-11): 12 x CamembertLayer(
          (attention): CamembertAttention(
            (self): CamembertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
        

In [46]:
def annotate_text(text, model, tokenizer, max_length=512, stride=128):

    encoding = tokenizer(
        text,
        return_offsets_mapping=True,
        return_overflowing_tokens=True,
        truncation=True,
        max_length=max_length,
        stride=stride,
        padding="max_length",
        return_tensors="pt",
    )

    offset_mapping = encoding.pop("offset_mapping")
    encoding.pop("overflow_to_sample_mapping")

    num_labels  = model.config.num_labels
    char_scores = np.zeros((len(text), num_labels), dtype=np.float32)
    char_counts = np.zeros(len(text), dtype=np.int32)

    for i in range(len(encoding["input_ids"])):
        chunk = {k: v[i].unsqueeze(0).to(device) for k, v in encoding.items()}
        with torch.no_grad():
            logits = model(**chunk).logits[0].cpu()
        probs = torch.softmax(logits, dim=-1).numpy()

        for token_idx, (start, end) in enumerate(offset_mapping[i]):
            s, e = int(start), int(end)
            if s == e:
                continue
            char_scores[s:e] += probs[token_idx]
            char_counts[s:e] += 1

    mask = char_counts > 0
    char_scores[mask] /= char_counts[mask, np.newaxis]

    import re
    id2label = model.config.id2label
    results  = []

    for m in re.finditer(r'\S+', text):
        s, e   = m.start(), m.end()
        window = char_scores[s:e]
        counts = char_counts[s:e]

        if counts.sum() == 0:
            label_id, score = 0, 0.0
        else:
            avg = window.mean(axis=0)
            label_id = int(avg.argmax())
            score = float(avg[label_id])

        results.append({
            "word": text[s:e],
            "label": id2label[label_id],
            "score": round(score, 4),
            "start": s,
            "end": e,
        })

    return results
 

In [43]:
import json
from tqdm import tqdm

def annotate_corpus(texts, ids, output_path):

    with open(output_path, "w", encoding="utf-8") as fout:
        for doc_id, text in tqdm(zip(ids, texts), total=len(texts)):
            try:
                annotations = annotate_text(text, model, tokenizer)
                record = {
                    "id": doc_id,
                    "text": text,
                    "annotations": annotations
                }
            except Exception as e:
                record = {"id": doc_id, "error": str(e)}

            fout.write(json.dumps(record, ensure_ascii=False) + "\n")

    print(f"Sauvegardé dans {output_path}")



In [47]:
textes = transcriptions["texte_nettoye"].to_list()
ids = transcriptions["id"]
output_path = "outputs/predictions_tot_disc.jsonl"

annotate_corpus(textes, ids, output_path)

  0%|          | 0/5936 [00:00<?, ?it/s]

100%|██████████| 5936/5936 [15:30<00:00,  6.38it/s]

Sauvegardé dans outputs/predictions_tot_disc.jsonl
